# Bangkok PM2.5 Forecasting — Visualization & EDA

**Input**: Preprocessed data from `data/gold/model_ready/`  
**Purpose**: Explore data quality, feature distributions, spatial/temporal patterns, and readiness for model training.

### Sections
| # | Section | Focus |
|---|---------|-------|
| 1 | Data Overview | Load splits, column audit, missing values |
| 2 | Data Completeness | Heatmap of available vs missing features |
| 3 | Station Map | Spatial distribution of 79 weather stations |
| 4 | Temporal Coverage | Timeline of data availability per station |
| 5 | Weather Distributions | Histograms & box plots for meteorological features |
| 6 | Seasonal Patterns | Monthly/seasonal variation in weather |
| 7 | Wind Analysis | Wind rose, U10/V10 scatter, directional patterns |
| 8 | Correlation Matrix | Inter-feature correlations |
| 9 | Feature Relationships | Scatter plots of key feature pairs |
| 10 | Train/Val/Test Split | Visual timeline of chronological split |
| 11 | PM2.5 & AQ Status | Current state + placeholder for when data arrives |

In [ ]:
import subprocess, sys

def _install_if_missing(packages: list[str]) -> None:
    """Install only packages that are not already importable."""
    missing = []
    import_map = {"scikit-learn": "sklearn", "pyarrow": "pyarrow"}
    for pkg in packages:
        mod = import_map.get(pkg, pkg.split("[")[0])
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    else:
        print("All packages already installed.")

_install_if_missing(["pandas", "pyarrow", "numpy", "matplotlib", "seaborn", "scikit-learn"])

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Plot style ──
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data/gold/model_ready")

COLORS = {
    "train": "#2196F3",
    "val": "#FF9800",
    "test": "#F44336",
    "weather": "#4CAF50",
    "aq": "#9C27B0",
    "hotspot": "#FF5722",
}

print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

---
## 1. Data Overview

In [ ]:
with open(DATA_DIR / "pipeline_manifest.json") as f:
    MANIFEST = json.load(f)

FEATURE_COLS = MANIFEST["feature_cols"]
TARGET_COL = MANIFEST["target_col"]

def load_split(name: str) -> pd.DataFrame:
    df = pd.read_parquet(DATA_DIR / f"{name}.parquet")
    df["date"] = pd.to_datetime(df["date"])
    df["split"] = name
    return df

df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")
df_all   = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f"{'Split':6s} {'Rows':>10s} {'Stations':>9s} {'Date range'}")
print("─" * 60)
for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"{name:6s} {len(df):>10,} {df['stationID'].nunique():>9d} {df['date'].min().date()} → {df['date'].max().date()}")
print(f"{'total':6s} {len(df_all):>10,} {df_all['stationID'].nunique():>9d} {df_all['date'].min().date()} → {df_all['date'].max().date()}")

---
## 2. Data Completeness Heatmap

In [ ]:
feature_groups = {
    "Weather (base)": ["temp_c", "humidity_pct", "pressure_hpa", "wind_ms", "wind_dir_deg",
                       "wind_u10", "wind_v10", "precipitation_mm", "cloud_cover_pct", "shortwave_radiation_wm2"],
    "Weather (lag)": [c for c in FEATURE_COLS if "_lag" in c and "pm2_5" not in c and "hotspot" not in c],
    "Weather (rolling)": [c for c in FEATURE_COLS if ("_rmean" in c or "_rstd" in c) and "pm2_5" not in c],
    "Air Quality": ["pm2_5_ugm3", "pm10_ugm3", "co_ugm3", "no2_ugm3", "o3_ugm3", "so2_ugm3"],
    "PM2.5 (lag/roll)": [c for c in FEATURE_COLS if "pm2_5" in c],
    "Hotspot": ["hotspot_count_th", "hotspot_count_mm", "hotspot_count_la", "hotspot_frp_sum", "transboundary_index"],
    "Temporal": ["doy_sin", "doy_cos", "month_sin", "month_cos"],
}

all_data_cols = [TARGET_COL] + FEATURE_COLS
existing_cols = [c for c in all_data_cols if c in df_all.columns]

completeness = pd.DataFrame({
    "column": existing_cols,
    "available_pct": [(1 - df_all[c].isna().mean()) * 100 for c in existing_cols],
})

def assign_group(col: str) -> str:
    for g, cols in feature_groups.items():
        if col in cols:
            return g
    return "Other"

completeness["group"] = completeness["column"].apply(assign_group)
completeness = completeness.sort_values(["group", "available_pct"], ascending=[True, False])

fig, ax = plt.subplots(figsize=(14, max(8, len(existing_cols) * 0.28)))

group_colors = {
    "Weather (base)": "#4CAF50", "Weather (lag)": "#66BB6A", "Weather (rolling)": "#81C784",
    "Air Quality": "#9C27B0", "PM2.5 (lag/roll)": "#BA68C8",
    "Hotspot": "#FF5722", "Temporal": "#2196F3", "Other": "#607D8B",
}

bar_colors = [group_colors.get(g, "#607D8B") for g in completeness["group"]]

ax.barh(range(len(completeness)), completeness["available_pct"].values, color=bar_colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(len(completeness)))
ax.set_yticklabels(completeness["column"].values, fontsize=8)
ax.set_xlabel("Data Availability (%)")
ax.set_title("Feature Completeness — All Splits Combined")
ax.axvline(100, color="green", linestyle="--", alpha=0.3, linewidth=1)
ax.axvline(0, color="red", linestyle="--", alpha=0.3, linewidth=1)
ax.set_xlim(-2, 105)
ax.invert_yaxis()

legend_patches = [mpatches.Patch(color=c, label=g) for g, c in group_colors.items() if g in completeness["group"].values]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "01_feature_completeness.png")
plt.show()

print("\n--- Summary ---")
for g in completeness["group"].unique():
    sub = completeness[completeness["group"] == g]
    avg = sub["available_pct"].mean()
    print(f"  {g:22s}: {avg:5.1f}% avg ({len(sub)} features)")

---
## 3. Station Map

In [ ]:
stations = df_all.groupby("stationID").agg(
    lat=("lat", "first"),
    lon=("lon", "first"),
    n_days=("date", "nunique"),
    date_min=("date", "min"),
    date_max=("date", "max"),
).reset_index()

BKK_LAT, BKK_LON = 13.7563, 100.5018

fig, ax = plt.subplots(figsize=(10, 10))

sc = ax.scatter(
    stations["lon"], stations["lat"],
    c=stations["n_days"], cmap="YlOrRd", s=40, edgecolor="black", linewidth=0.4, alpha=0.85,
    zorder=3,
)
plt.colorbar(sc, ax=ax, label="Days of data", shrink=0.7)

ax.scatter(BKK_LON, BKK_LAT, marker="*", s=300, c="blue", edgecolor="white", linewidth=1.5, zorder=5, label="Bangkok center")

circle = plt.Circle((BKK_LON, BKK_LAT), 800 / 111.32, fill=False, linestyle="--", color="blue", alpha=0.3, linewidth=1.5)
ax.add_patch(circle)
ax.text(BKK_LON + 0.3, BKK_LAT + 800 / 111.32 - 0.3, "800 km radius", fontsize=8, color="blue", alpha=0.5)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Weather Station Locations ({len(stations)} stations)")
ax.legend(loc="upper left")
ax.set_aspect("equal")

lat_pad = max((stations["lat"].max() - stations["lat"].min()) * 0.1, 1)
lon_pad = max((stations["lon"].max() - stations["lon"].min()) * 0.1, 1)
ax.set_xlim(stations["lon"].min() - lon_pad, stations["lon"].max() + lon_pad)
ax.set_ylim(stations["lat"].min() - lat_pad, stations["lat"].max() + lat_pad)

ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "02_station_map.png")
plt.show()

print(f"Station lat range: {stations['lat'].min():.2f} — {stations['lat'].max():.2f}")
print(f"Station lon range: {stations['lon'].min():.2f} — {stations['lon'].max():.2f}")
print(f"Days per station : {stations['n_days'].min()} — {stations['n_days'].max()} (median {stations['n_days'].median():.0f})")

---
## 4. Temporal Coverage

In [ ]:
stations_sorted = stations.sort_values("n_days", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, max(6, len(stations_sorted) * 0.12)))

for i, row in stations_sorted.iterrows():
    ax.barh(
        i,
        (row["date_max"] - row["date_min"]).days,
        left=mdates.date2num(row["date_min"]),
        height=0.7,
        color=COLORS["weather"],
        alpha=0.7,
        edgecolor="white",
        linewidth=0.3,
    )

ax.set_yticks(range(len(stations_sorted)))
ax.set_yticklabels(stations_sorted["stationID"].values, fontsize=5)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())

for split_name, split_df, color in [("train", df_train, COLORS["train"]), ("val", df_val, COLORS["val"]), ("test", df_test, COLORS["test"])]:
    boundary = mdates.date2num(split_df["date"].min())
    ax.axvline(boundary, color=color, linestyle="--", linewidth=1.5, alpha=0.7, label=f"{split_name} start")

ax.set_xlabel("Date")
ax.set_title("Temporal Coverage per Station")
ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "03_temporal_coverage.png")
plt.show()

---
## 5. Weather Feature Distributions

In [ ]:
weather_base = ["temp_c", "humidity_pct", "pressure_hpa", "wind_ms",
                "precipitation_mm", "cloud_cover_pct", "shortwave_radiation_wm2"]
available_weather = [c for c in weather_base if c in df_all.columns and df_all[c].notna().sum() > 100]

n = len(available_weather)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(available_weather):
    ax = axes[i]
    data = df_all[col].dropna()
    ax.hist(data, bins=60, color=COLORS["weather"], alpha=0.7, edgecolor="white", linewidth=0.3)
    ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1, label=f"mean={data.mean():.1f}")
    ax.axvline(data.median(), color="orange", linestyle="--", linewidth=1, label=f"med={data.median():.1f}")
    ax.set_title(col)
    ax.legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Weather Feature Distributions (all splits)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "04_weather_distributions.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(available_weather), figsize=(3 * len(available_weather), 5))

for i, col in enumerate(available_weather):
    ax = axes[i]
    box_data = [df_train[col].dropna(), df_val[col].dropna(), df_test[col].dropna()]
    bp = ax.boxplot(box_data, labels=["Train", "Val", "Test"], patch_artist=True, widths=0.6)
    for patch, color in zip(bp["boxes"], [COLORS["train"], COLORS["val"], COLORS["test"]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

fig.suptitle("Feature Distribution by Split — Checking for Data Drift", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "05_split_drift_boxplots.png")
plt.show()

---
## 6. Seasonal Patterns

In [ ]:
df_all["month"] = df_all["date"].dt.month
df_all["year"] = df_all["date"].dt.year

MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

season_features = ["temp_c", "humidity_pct", "wind_ms", "precipitation_mm"]
available_season = [c for c in season_features if c in df_all.columns and df_all[c].notna().sum() > 100]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(available_season):
    ax = axes[i]
    monthly = df_all.groupby("month")[col].agg(["mean", "std"]).reindex(range(1, 13))
    ax.fill_between(range(1, 13), monthly["mean"] - monthly["std"], monthly["mean"] + monthly["std"],
                    alpha=0.2, color=COLORS["weather"])
    ax.plot(range(1, 13), monthly["mean"], "o-", color=COLORS["weather"], linewidth=2, markersize=6)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS, fontsize=9)
    ax.set_title(f"{col} — Monthly Average ± 1σ")
    ax.set_ylabel(col)

    # Thai season backgrounds
    ax.axvspan(1.5, 4.5, alpha=0.06, color="red", label="Burning")
    ax.axvspan(4.5, 9.5, alpha=0.06, color="green", label="Monsoon")
    ax.axvspan(9.5, 12.5, alpha=0.06, color="blue", label="Cool")
    ax.axvspan(0.5, 1.5, alpha=0.06, color="blue")

for j in range(len(available_season), len(axes)):
    axes[j].set_visible(False)

handles = [
    mpatches.Patch(color="red", alpha=0.15, label="Burning (Feb-Apr)"),
    mpatches.Patch(color="green", alpha=0.15, label="Monsoon (May-Sep)"),
    mpatches.Patch(color="blue", alpha=0.15, label="Cool (Oct-Jan)"),
]
fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=10, bbox_to_anchor=(0.5, 1.02))

fig.suptitle("Seasonal Weather Patterns — Thai Pollution Seasons", fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig(PLOT_DIR / "06_seasonal_patterns.png")
plt.show()

In [ ]:
yearly_features = ["temp_c", "humidity_pct", "wind_ms"]
available_yearly = [c for c in yearly_features if c in df_all.columns and df_all[c].notna().sum() > 100]

fig, axes = plt.subplots(1, len(available_yearly), figsize=(6 * len(available_yearly), 5))
if len(available_yearly) == 1:
    axes = [axes]

for i, col in enumerate(available_yearly):
    ax = axes[i]
    for year in sorted(df_all["year"].unique()):
        yd = df_all[df_all["year"] == year].groupby("month")[col].mean()
        ax.plot(yd.index, yd.values, alpha=0.5, linewidth=1, label=str(year))
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS, fontsize=8)
    ax.set_title(f"{col} — Year-over-Year")
    ax.legend(fontsize=6, ncol=2)

plt.tight_layout()
plt.savefig(PLOT_DIR / "07_yearly_comparison.png")
plt.show()

---
## 7. Wind Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 7a. Wind U10 vs V10 scatter
ax = axes[0]
sample = df_all[["wind_u10", "wind_v10"]].dropna().sample(n=min(5000, len(df_all)), random_state=42)
ax.scatter(sample["wind_u10"], sample["wind_v10"], alpha=0.15, s=3, color="steelblue")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("U10 (east-west, m/s)")
ax.set_ylabel("V10 (north-south, m/s)")
ax.set_title("Wind Components")
ax.set_aspect("equal")

# 7b. Wind speed distribution
ax = axes[1]
wind_data = df_all["wind_ms"].dropna()
ax.hist(wind_data, bins=50, color="steelblue", alpha=0.7, edgecolor="white")
ax.axvline(wind_data.mean(), color="red", linestyle="--", label=f"mean={wind_data.mean():.1f} m/s")
ax.set_xlabel("Wind speed (m/s)")
ax.set_ylabel("Count")
ax.set_title("Wind Speed Distribution")
ax.legend()

# 7c. Wind direction (simple polar histogram)
ax_polar = fig.add_subplot(1, 3, 3, projection="polar")
axes[2].set_visible(False)
wind_dirs = df_all["wind_dir_deg"].dropna()
bins_deg = np.linspace(0, 360, 37)
counts, _ = np.histogram(wind_dirs, bins=bins_deg)
theta = np.deg2rad((bins_deg[:-1] + bins_deg[1:]) / 2)
width = np.deg2rad(10)
ax_polar.bar(theta, counts, width=width, color="steelblue", alpha=0.7, edgecolor="white", linewidth=0.3)
ax_polar.set_theta_zero_location("N")
ax_polar.set_theta_direction(-1)
ax_polar.set_title("Wind Direction Rose", pad=20)

plt.tight_layout()
plt.savefig(PLOT_DIR / "08_wind_analysis.png")
plt.show()

In [ ]:
wind_monthly = df_all.groupby("month")[["wind_u10", "wind_v10"]].mean().reindex(range(1, 13))

fig, ax = plt.subplots(figsize=(8, 8))

for m in range(1, 13):
    u = wind_monthly.loc[m, "wind_u10"]
    v = wind_monthly.loc[m, "wind_v10"]
    ax.annotate("", xy=(u, v), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=plt.cm.hsv(m / 12), linewidth=2))
    ax.text(u * 1.1, v * 1.1, MONTH_LABELS[m - 1], fontsize=9, ha="center", va="center",
            fontweight="bold", color=plt.cm.hsv(m / 12))

ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Mean U10 (eastward m/s) →")
ax.set_ylabel("Mean V10 (northward m/s) →")
ax.set_title("Monthly Mean Wind Vectors\n(arrows from origin = wind blowing towards)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

lim = max(abs(wind_monthly.values.flatten().max()), abs(wind_monthly.values.flatten().min())) * 1.4
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)

ax.annotate("NW influence\n(Burning season)", xy=(-lim * 0.7, lim * 0.7), fontsize=8, color="gray", ha="center")
ax.annotate("SW monsoon", xy=(-lim * 0.7, -lim * 0.7), fontsize=8, color="gray", ha="center")
ax.annotate("NE monsoon", xy=(lim * 0.7, lim * 0.7), fontsize=8, color="gray", ha="center")

plt.tight_layout()
plt.savefig(PLOT_DIR / "09_wind_vectors_monthly.png")
plt.show()

---
## 8. Correlation Matrix

In [ ]:
corr_cols = [c for c in weather_base + ["wind_u10", "wind_v10"] if c in df_all.columns and df_all[c].notna().mean() > 0.5]

corr_matrix = df_all[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
    annot_kws={"size": 9},
)
ax.set_title("Weather Feature Correlations")

plt.tight_layout()
plt.savefig(PLOT_DIR / "10_correlation_matrix.png")
plt.show()

print("\n--- Strong correlations (|r| > 0.7) ---")
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            print(f"  {corr_matrix.index[i]:30s} ↔ {corr_matrix.columns[j]:30s} : r={r:+.3f}")

In [ ]:
lag_cols = [c for c in FEATURE_COLS if ("_lag" in c or "_rmean" in c or "_rstd" in c) and df_all[c].notna().mean() > 0.5]

if len(lag_cols) > 2:
    lag_corr = df_all[lag_cols].corr()

    fig, ax = plt.subplots(figsize=(max(10, len(lag_cols) * 0.6), max(8, len(lag_cols) * 0.5)))
    sns.heatmap(
        lag_corr, annot=False, cmap="RdBu_r", center=0,
        square=True, linewidths=0.2, ax=ax, vmin=-1, vmax=1,
        xticklabels=True, yticklabels=True,
    )
    ax.set_title("Lag & Rolling Feature Correlations")
    ax.tick_params(labelsize=6)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / "11_lag_rolling_correlation.png")
    plt.show()
else:
    print(f"Only {len(lag_cols)} lag/rolling columns with data — skipping.")

---
## 9. Feature Relationships — Key Scatter Plots

In [ ]:
scatter_pairs = [
    ("temp_c", "humidity_pct"),
    ("temp_c", "pressure_hpa"),
    ("wind_ms", "precipitation_mm"),
    ("shortwave_radiation_wm2", "cloud_cover_pct"),
    ("temp_c", "shortwave_radiation_wm2"),
    ("humidity_pct", "precipitation_mm"),
]

valid_pairs = [(x, y) for x, y in scatter_pairs
               if x in df_all.columns and y in df_all.columns
               and df_all[x].notna().sum() > 100 and df_all[y].notna().sum() > 100]

ncols = 3
nrows = (len(valid_pairs) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()

for i, (x_col, y_col) in enumerate(valid_pairs):
    ax = axes[i]
    sub = df_all[[x_col, y_col]].dropna().sample(n=min(5000, len(df_all)), random_state=42)
    ax.scatter(sub[x_col], sub[y_col], alpha=0.1, s=3, color="steelblue")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    r = sub[x_col].corr(sub[y_col])
    ax.set_title(f"{x_col} vs {y_col} (r={r:.2f})")

for j in range(len(valid_pairs), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Key Feature Relationships", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "12_feature_scatter.png")
plt.show()

---
## 10. Train / Val / Test Split Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"height_ratios": [1, 3]})

# 10a. Top: split timeline bar
ax = axes[0]
for split_name, split_df, color in [
    ("train", df_train, COLORS["train"]),
    ("val", df_val, COLORS["val"]),
    ("test", df_test, COLORS["test"]),
]:
    d_min = mdates.date2num(split_df["date"].min())
    d_max = mdates.date2num(split_df["date"].max())
    ax.barh(0, d_max - d_min, left=d_min, height=0.5, color=color, alpha=0.8, label=f"{split_name} ({len(split_df):,} rows)")
    mid = d_min + (d_max - d_min) / 2
    ax.text(mid, 0, f"{split_name}\n{split_df['date'].min().date()} → {split_df['date'].max().date()}",
            ha="center", va="center", fontsize=8, fontweight="bold", color="white")

ax.set_yticks([])
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_title("Chronological Data Split")
ax.legend(loc="upper left", fontsize=9)

# 10b. Bottom: daily record count per split
ax = axes[1]
for split_name, split_df, color in [
    ("train", df_train, COLORS["train"]),
    ("val", df_val, COLORS["val"]),
    ("test", df_test, COLORS["test"]),
]:
    daily = split_df.groupby("date").size()
    ax.plot(daily.index, daily.values, color=color, alpha=0.7, linewidth=0.5, label=split_name)

ax.set_xlabel("Date")
ax.set_ylabel("Records / day")
ax.set_title("Daily Record Count")
ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / "13_train_val_test_split.png")
plt.show()

print("\n--- Split Statistics ---")
print(f"  Train : {len(df_train):>10,} rows  ({len(df_train)/len(df_all)*100:.1f}%)  {df_train['date'].min().date()} → {df_train['date'].max().date()}")
print(f"  Val   : {len(df_val):>10,} rows  ({len(df_val)/len(df_all)*100:.1f}%)  {df_val['date'].min().date()} → {df_val['date'].max().date()}")
print(f"  Test  : {len(df_test):>10,} rows  ({len(df_test)/len(df_all)*100:.1f}%)  {df_test['date'].min().date()} → {df_test['date'].max().date()}")

---
## 11. PM2.5 & Air Quality Status

Current status check — visualizations activate when AQ data becomes available.

In [ ]:
aq_cols = ["pm2_5_ugm3", "pm10_ugm3", "co_ugm3", "no2_ugm3", "o3_ugm3", "so2_ugm3"]
aq_available = {c: df_all[c].notna().sum() for c in aq_cols if c in df_all.columns}

print("=" * 50)
print("  AIR QUALITY DATA STATUS")
print("=" * 50)
for col, cnt in aq_available.items():
    pct = cnt / len(df_all) * 100
    status = "AVAILABLE" if pct > 1 else "MISSING"
    print(f"  {col:20s}: {cnt:>10,} / {len(df_all):,} ({pct:5.1f}%) — {status}")

has_pm25 = aq_available.get("pm2_5_ugm3", 0) > 100

In [ ]:
if has_pm25:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # PM2.5 time series
    ax = axes[0, 0]
    daily_pm25 = df_all.groupby("date")["pm2_5_ugm3"].mean()
    ax.plot(daily_pm25.index, daily_pm25.values, linewidth=0.5, alpha=0.7, color="purple")
    ax.axhline(50, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Unhealthy (50 µg/m³)")
    ax.axhline(25, color="orange", linestyle="--", linewidth=1, alpha=0.5, label="WHO guideline (25 µg/m³)")
    ax.set_title("Daily Mean PM2.5")
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.legend(fontsize=8)

    # PM2.5 distribution
    ax = axes[0, 1]
    pm25_vals = df_all["pm2_5_ugm3"].dropna()
    ax.hist(pm25_vals, bins=80, color="purple", alpha=0.7, edgecolor="white")
    ax.axvline(50, color="red", linestyle="--", label="Unhealthy")
    ax.set_title("PM2.5 Distribution")
    ax.set_xlabel("PM2.5 (µg/m³)")
    ax.legend()

    # PM2.5 monthly
    ax = axes[1, 0]
    monthly_pm = df_all.groupby("month")["pm2_5_ugm3"].agg(["mean", "std"]).reindex(range(1, 13))
    ax.bar(range(1, 13), monthly_pm["mean"], color="purple", alpha=0.6, yerr=monthly_pm["std"], capsize=3)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS)
    ax.set_title("PM2.5 — Monthly Average")
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.axhline(50, color="red", linestyle="--", alpha=0.3)

    # PM2.5 vs temperature
    ax = axes[1, 1]
    sub = df_all[["pm2_5_ugm3", "temp_c"]].dropna().sample(n=min(5000, len(df_all)), random_state=42)
    ax.scatter(sub["temp_c"], sub["pm2_5_ugm3"], alpha=0.1, s=3, color="purple")
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_title(f"PM2.5 vs Temperature (r={sub['temp_c'].corr(sub['pm2_5_ugm3']):.2f})")

    plt.tight_layout()
    plt.savefig(PLOT_DIR / "14_pm25_analysis.png")
    plt.show()
else:
    print("\n⚠ PM2.5 data is not yet available.")
    print("  Action: Fix AQ ingestion in bangkok_environmental_ingestion.ipynb")
    print("          then re-run preprocessing_pipeline.ipynb")
    print("\n  When PM2.5 data arrives, this cell will produce:")
    print("    • Daily mean PM2.5 time series with WHO/unhealthy thresholds")
    print("    • PM2.5 distribution histogram")
    print("    • Monthly average PM2.5 (burning season peaks)")
    print("    • PM2.5 vs temperature scatter")

In [ ]:
hs_cols = ["hotspot_count_th", "hotspot_count_mm", "hotspot_count_la", "hotspot_frp_sum", "transboundary_index"]
hs_available = {c: df_all[c].notna().sum() for c in hs_cols if c in df_all.columns}
has_hotspot = any(v > 100 for v in hs_available.values())

print("=" * 50)
print("  HOTSPOT DATA STATUS")
print("=" * 50)
for col, cnt in hs_available.items():
    pct = cnt / len(df_all) * 100
    status = "AVAILABLE" if pct > 1 else "MISSING"
    print(f"  {col:25s}: {cnt:>10,} / {len(df_all):,} ({pct:5.1f}%) — {status}")

if has_hotspot:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    for hs_col in ["hotspot_count_th", "hotspot_count_mm", "hotspot_count_la"]:
        if hs_col in df_all.columns and df_all[hs_col].notna().sum() > 0:
            daily = df_all.groupby("date")[hs_col].mean()
            ax.plot(daily.index, daily.values, linewidth=0.5, alpha=0.7, label=hs_col.replace("hotspot_count_", "").upper())
    ax.set_title("Daily Hotspot Count by Country")
    ax.legend()

    ax = axes[1]
    if "hotspot_frp_sum" in df_all.columns:
        monthly_frp = df_all.groupby("month")["hotspot_frp_sum"].mean().reindex(range(1, 13))
        ax.bar(range(1, 13), monthly_frp.values, color=COLORS["hotspot"], alpha=0.7)
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels(MONTH_LABELS)
        ax.set_title("Monthly Mean FRP Sum")

    ax = axes[2]
    if "transboundary_index" in df_all.columns:
        tbi = df_all.groupby("date")["transboundary_index"].mean()
        ax.plot(tbi.index, tbi.values, linewidth=0.5, color=COLORS["hotspot"], alpha=0.7)
        ax.set_title("Transboundary Index (daily mean)")

    plt.tight_layout()
    plt.savefig(PLOT_DIR / "15_hotspot_analysis.png")
    plt.show()
else:
    print("\n⚠ Hotspot data is not yet available.")
    print("  Action: Integrate NASA FIRMS VIIRS data")
    print("\n  When hotspot data arrives, this cell will produce:")
    print("    • Daily hotspot counts by country (TH, MM, LA)")
    print("    • Monthly mean FRP sum")
    print("    • Transboundary Index time series")

---
## 12. Summary Statistics

In [ ]:
print("\n" + "=" * 70)
print("  VISUALIZATION SUMMARY")
print("=" * 70)

print(f"\n--- Dataset ---")
print(f"  Total records : {len(df_all):,}")
print(f"  Stations      : {df_all['stationID'].nunique()}")
print(f"  Date range    : {df_all['date'].min().date()} → {df_all['date'].max().date()}")
print(f"  Features      : {len(FEATURE_COLS)} columns")

print(f"\n--- Data Availability ---")
n_total = len(df_all)
weather_avail = df_all["temp_c"].notna().sum() / n_total * 100
pm25_avail = df_all["pm2_5_ugm3"].notna().sum() / n_total * 100 if "pm2_5_ugm3" in df_all.columns else 0
hs_avail = df_all["hotspot_count_th"].notna().sum() / n_total * 100 if "hotspot_count_th" in df_all.columns else 0
print(f"  Weather    : {weather_avail:5.1f}%")
print(f"  PM2.5 (AQ) : {pm25_avail:5.1f}%")
print(f"  Hotspot    : {hs_avail:5.1f}%")

print(f"\n--- Plots Saved ---")
for f in sorted(PLOT_DIR.iterdir()):
    if f.suffix == ".png":
        print(f"  {f.name:40s} ({f.stat().st_size / 1e3:.0f} KB)")

print(f"\n--- Next Steps ---")
if pm25_avail < 1:
    print("  [CRITICAL] Fix AQ ingestion → re-run preprocessing → re-run this notebook")
if hs_avail < 1:
    print("  [PENDING]  Integrate NASA FIRMS hotspot data")
if weather_avail > 99 and pm25_avail > 50 and hs_avail > 50:
    print("  [READY]    All data sources available — proceed to model_training.ipynb")

print("\n" + "=" * 70)